# Week 2 Debate Simulation
**Author:** Mark Byron (byronmaexp13)

A multi-agent debate simulation using the OpenRouter API with Google Gemini 3.5 Flash. 
Features a moderator and two debating agents with configurable turns.

In [ ]:
"""
Title: Multi-Agent Debate Simulation (Secure Edition)
Summary: This script dynamically loads the OpenRouter API key from your 
local .env file. No credentials are hardcoded.
"""

import time
import json
import textwrap
import requests
import os
from dotenv import load_dotenv

# 1. Load variables from your existing .env file
load_dotenv()

# 2. Configuration: Key is pulled from system environment (from .env)
API_KEY = os.getenv("OPENROUTER_API_KEY")
if not API_KEY:
    raise ValueError("API_KEY not found in .env file! Please check your configuration.")

API_URL = "https://openrouter.ai/api/v1/chat/completions"

conversation_history = [
    {"role": "system", "content": "You are a participant in a 3-way conversation."}
]

personas = {
    "Alex": "You are snaky, argumentative, and cynical. Keep responses under 2 sentences.",
    "Blake": "You are optimistic, project-focused, and eager to move forward. Keep responses under 2 sentences.",
    "Charlie": "You are cautious, detailed, and slow to commit. Keep responses under 2 sentences.",
    "Moderator": "You are a neutral moderator. Keep responses under 2 sentences."
}

# 3. API Interaction: Using the verified v3.5-flash model
def call_openrouter(messages):
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json",
        "HTTP-Referer": "https://localhost",
        "X-Title": "Multi-Agent Simulation"
    }
    
    payload = {
        "model": "google/gemini-3.5-flash", 
        "messages": messages
    }
    
    response = requests.post(API_URL, headers=headers, json=payload)
    
    if response.status_code != 200:
        raise Exception(f"API Error {response.status_code}: {response.text}")
        
    return response.json()['choices'][0]['message']['content']

# 4. Simulation Logic
def run_simulation(turns=3):
    for i in range(turns):
        for speaker in ["Blake", "Charlie", "Alex", "Moderator"]:
            system_msg = {"role": "system", "content": personas[speaker]}
            full_context = [system_msg] + conversation_history[-6:]
            
            try:
                reply = call_openrouter(full_context)
            except Exception as e:
                reply = f"[Error: {e}]"
            
            conversation_history.append({"role": "assistant", "name": speaker, "content": reply})
            print(f"{speaker}: {reply}")
            time.sleep(1)

def summarize_debate():
    summary_request = "Summarize the key points of contention and current status."
    messages = [{"role": "user", "content": summary_request}] + conversation_history
    summary = call_openrouter(messages)
    return textwrap.fill(summary, width=70)

def save_for_pr(filename="debate_log.json"):
    with open(filename, "w") as f:
        json.dump(conversation_history, f, indent=4)
    print(f"\n[Saved] Log exported to {filename}.")

if __name__ == "__main__":
    run_simulation(turns=2)
    print("\n" + "="*30)
    print("EXECUTIVE SUMMARY")
    print("="*30)
    print(summarize_debate())
    save_for_pr()

In [ ]:
# --- DIAGNOSTIC CELL: Verify available models ---
import requests
import os
from dotenv import load_dotenv

load_dotenv() # Load your API key from .env
API_KEY = os.getenv("OPENROUTER_API_KEY")

if API_KEY:
    headers = {"Authorization": f"Bearer {API_KEY}"}
    try:
        response = requests.get("https://openrouter.ai/api/v1/models", headers=headers)
        data = response.json()
        gemini_models = [m['id'] for m in data['data'] if 'google' in m['id'] and 'gemini' in m['id']]
        
        print("Verified Gemini Models available for your key:")
        for m in gemini_models[:10]:
            print(f"- {m}")
    except Exception as e:
        print(f"Could not fetch model list: {e}")
else:
    print("API_KEY not found in .env. Please configure your .env file.")